# Calibration analysis (Black / Heston / SVCJ) — thesis figures & tables

This notebook turns `calibration_results.xlsx` into the figures and tables used in the thesis calibration-results section.

**Design choices**
- **Snapshot is the sampling unit**. Where we report confidence intervals for average errors, we bootstrap **over snapshots** (not over individual quotes).
- Errors are measured in **coin premium units** (inverse option numeraire).
- We report both **price-space errors** (RMSE/MAE) and **microstructure-aware** diagnostics (within-spread fractions, error/spread, etc.).

> If you moved the Excel file, update `DATA_PATH` in the next cell.


In [1]:
from pathlib import Path
import sys

from IPython.display import display

NOTEBOOK_ROOT = Path.cwd().resolve()
if (NOTEBOOK_ROOT / "_lib").exists():
    NOTEBOOK_DIR = NOTEBOOK_ROOT
elif (NOTEBOOK_ROOT / "notebooks" / "_lib").exists():
    NOTEBOOK_DIR = NOTEBOOK_ROOT / "notebooks"
else:
    raise FileNotFoundError(f"Could not locate notebooks/_lib from {NOTEBOOK_ROOT}")

if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from _lib import chapter3_analysis as analysis
from _lib.common import ensure_existing_path, locate_notebook_paths

RNG = analysis.initialize_notebook_defaults()
PATHS = locate_notebook_paths(NOTEBOOK_DIR)
DATA_PATH = "calibration_results_reg_1.xlsx"
MODEL_SPECS = analysis.MODEL_SPECS
COLORS = analysis.COLORS
FIGDIMS = analysis.FIGDIMS
DATA_FILE = ensure_existing_path(Path(DATA_PATH), search_dirs=[PATHS.excel_dir, PATHS.project_root, PATHS.notebook_dir])


/opt/miniconda3/lib/python3.13/site-packages/kaleido/_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.




In [2]:
state = analysis.build_analysis_state(DATA_FILE, rng=RNG)

black_params = state.black_params
heston_params = state.heston_params
svcj_params = state.svcj_params
train_data = state.train_data
test_data = state.test_data

print("Loaded from:", DATA_FILE)
print(" - black_params:", black_params.shape)
print(" - heston_params:", heston_params.shape)
print(" - svcj_params:", svcj_params.shape)
print(" - train_data:", train_data.shape)
print(" - test_data :", test_data.shape)

display(black_params.head(3))


Loaded from: /Users/nikitamahbub/Desktop/uni/thesis/SVCJ-Deribit-Inverse-Options-Pricer-clean/excel files/calibration_results_reg_1.xlsx
 - black_params: (776, 16)
 - heston_params: (776, 20)
 - svcj_params: (776, 25)
 - train_data: (248526, 34)
 - test_data : (107026, 34)


/Users/nikitamahbub/Desktop/uni/thesis/SVCJ-Deribit-Inverse-Options-Pricer-clean/notebooks/_lib/chapter3_analysis.py:576: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

/Users/nikitamahbub/Desktop/uni/thesis/SVCJ-Deribit-Inverse-Options-Pricer-clean/notebooks/_lib/chapter3_analysis.py:582: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

/Users/nikitamahbub/Desktop/uni/thesis/SVCJ-Deribit-Inverse-Options-Pricer-clean/notebooks/_lib/chapter3_analysis.py:576: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current b

,timestamp,currency,success,message,nfev,rmse_fit,mae_fit,rmse_train,mae_train,rmse_test,mae_test,n_options_total,n_train,n_test,random_seed,sigma
0,2025-12-30 17:31:15+00:00,BTC,True,`ftol` termination condition is satisfied.,5,0.005825,0.003963,0.005825,0.003963,0.004958,0.003629,398,278,120,123,0.433832
1,2025-12-30 21:17:51+00:00,BTC,True,`ftol` termination condition is satisfied.,4,0.005975,0.004278,0.005975,0.004278,0.006321,0.003966,392,274,118,124,0.431057
2,2025-12-31 09:18:28+00:00,BTC,True,`ftol` termination condition is satisfied.,5,0.005681,0.003941,0.005681,0.003941,0.004456,0.003473,391,273,118,125,0.445090


## 1) Build snapshot-level “results” tables (metrics + convergence + parameters)

We consolidate the three model-specific parameter sheets into a common long format:

- One row per *(snapshot, currency, model)*  
- With: train/test RMSE & MAE, success flag, solver message, `nfev`, and calibrated parameters.


In [3]:
results_long = state.results_long
results_ok = state.results_ok

display(results_long.head(6))
print("Currencies:", results_long["currency"].unique())


,timestamp,currency,success,message,nfev,rmse_fit,mae_fit,rmse_train,mae_train,rmse_test,mae_test,n_options_total,n_train,n_test,random_seed,sigma,model,kappa,theta,sigma_v,rho,v0,lam,ell_y,sigma_y,ell_v,rho_j
0,2025-12-30 17:31:15+00:00,BTC,True,`ftol` termination condition is satisfied.,5,0.005825,0.003963,0.005825,0.003963,0.004958,0.003629,398,278,120,123,0.433832,Black,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-12-30 17:31:15+00:00,BTC,True,`ftol` termination condition is satisfied.,30,0.002431,0.001201,0.002431,0.001201,0.001521,0.001009,398,278,120,123,NaN,Heston,5.731788,0.267392,1.755150,-0.214405,0.146301,NaN,NaN,NaN,NaN,NaN
2,2025-12-30 17:31:15+00:00,BTC,True,`ftol` termination condition is satisfied.,55,0.002106,0.000700,0.002106,0.000700,0.001286,0.000503,398,278,120,123,NaN,SVCJ,3.438955,0.095675,0.514601,-0.202938,0.118918,1.184894,-0.064701,0.204992,0.407451,-0.073967
3,2025-12-30 21:17:51+00:00,BTC,True,`ftol` termination condition is satisfied.,4,0.005975,0.004278,0.005975,0.004278,0.006321,0.003966,392,274,118,124,0.431057,Black,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025-12-30 21:17:51+00:00,BTC,True,`ftol` termination condition is satisfied.,14,0.002594,0.001377,0.002594,0.001377,0.003193,0.001262,392,274,118,124,NaN,Heston,6.252055,0.263021,1.817336,-0.197904,0.142408,NaN,NaN,NaN,NaN,NaN
5,2025-12-30 21:17:51+00:00,BTC,True,`ftol` termination condition is satisfied.,11,0.002395,0.000868,0.002395,0.000868,0.002782,0.000769,392,274,118,124,NaN,SVCJ,4.182341,0.084769,0.352466,-0.246416,0.113564,1.399337,-0.060607,0.190016,0.420458,-0.060590


Currencies: ['BTC' 'ETH']


## 2) Option-level metrics derived from `train_data` and `test_data`

The parameter sheets already contain RMSE/MAE train/test, but option-level data lets us compute
additional diagnostics (spread-relative errors, within-spread fractions, bucket analyses, etc.).


In [4]:
opt_metrics = state.opt_metrics

display(opt_metrics.head(6))
print("Option-derived snapshot metrics:", opt_metrics.shape)


,currency,snapshot_ts,n,mse,mae,within_spread,within_half_spread,abs_err_over_spread,smape,mean_price,rmse,rmse_over_mean_price,model,split
0,BTC,2025-12-30 17:31:15+00:00,120,0.000025,0.003629,0.225000,0.150000,3.140859,0.248498,0.083687,0.004958,0.059246,Black,test
1,BTC,2025-12-30 17:31:15+00:00,278,0.000034,0.003963,0.291367,0.233813,2.927041,0.215789,0.121861,0.005825,0.047797,Black,train
2,BTC,2025-12-30 17:31:15+00:00,120,0.000002,0.001009,0.658333,0.408333,1.157900,0.151182,0.083687,0.001521,0.018175,Heston,test
3,BTC,2025-12-30 17:31:15+00:00,278,0.000006,0.001201,0.744604,0.500000,0.887746,0.111333,0.121861,0.002431,0.019947,Heston,train
4,BTC,2025-12-30 17:31:15+00:00,120,0.000002,0.000503,0.925000,0.808333,0.329393,0.028150,0.083687,0.001286,0.015363,SVCJ,test
5,BTC,2025-12-30 17:31:15+00:00,278,0.000004,0.000700,0.960432,0.874101,0.247692,0.019607,0.121861,0.002106,0.017280,SVCJ,train


Option-derived snapshot metrics: (4638, 14)


## 3) Bootstrap helpers (snapshot-level)

We treat each snapshot as one observation. For a snapshot-level series \(m_t\), we report:
- mean + 95% bootstrap CI for the mean (percentile bootstrap),
- median, quartiles, standard deviation, and sample size.


In [5]:
def bootstrap_mean_ci(values, n_boot=3000, alpha=0.05, rng=RNG):
    return analysis.bootstrap_mean_ci(values, n_boot=n_boot, alpha=alpha, rng=rng)


def summarize_snapshot_series(values, n_boot=3000):
    return analysis.summarize_snapshot_series(values, n_boot=n_boot, rng=RNG)


## 4) Plot helpers (time-series and boxplots)

We keep the same **2×2 subplot layout** you already use:

1) RMSE (all models)  
2) MAE (all models)  
3) RMSE (Heston vs SVCJ)  
4) MAE (Heston vs SVCJ)


In [6]:
add_line = analysis.add_line
add_box = analysis.add_box
add_subplot_legend = analysis.add_subplot_legend
plot_error_timeseries = analysis.plot_error_timeseries
plot_error_boxplots = analysis.plot_error_boxplots


## 5) Summary tables (errors + CI bands + incremental gains)

This produces:
- per-model summary stats (mean + 95% CI, median, quartiles, etc.)
- incremental gains and win rates for:
  - Heston vs Black
  - SVCJ vs Heston


In [7]:
def error_summary_table(results_long_df, currency, split="test", n_boot=3000):
    return analysis.error_summary_table(results_long_df, currency, split=split, n_boot=n_boot, rng=RNG)


## 6) Convergence diagnostics (success, termination messages, nfev)

We summarize by *(currency, model)*:
- number of snapshots
- success rate
- `nfev` distribution (median / p90 / max)
- how often the solver hit the max evaluation cap (detected from termination message)


In [8]:
convergence_table = analysis.convergence_table

display(convergence_table(results_long))


,currency,model,n_snapshots,success_rate,nfev_median,nfev_mean,nfev_p90,nfev_max,hit_cap_rate,top_message_1,top_message_2,top_message_3
0,BTC,Black,388,1.000000,4.0,4.126289,5.0,6,0.000000,`ftol` termination condition is satisfied.,Both `ftol` and `xtol` termination conditions ...,`xtol` termination condition is satisfied.
1,BTC,Heston,388,1.000000,12.0,12.858247,19.0,47,0.000000,`ftol` termination condition is satisfied.,NaN,NaN
2,BTC,SVCJ,388,0.984536,17.0,27.932990,55.0,250,0.015464,`ftol` termination condition is satisfied.,The maximum number of function evaluations is ...,`xtol` termination condition is satisfied.
3,ETH,Black,388,1.000000,4.0,4.149485,5.0,6,0.000000,`ftol` termination condition is satisfied.,NaN,NaN
4,ETH,Heston,388,1.000000,9.0,10.868557,16.0,73,0.000000,`ftol` termination condition is satisfied.,Both `ftol` and `xtol` termination conditions ...,NaN
5,ETH,SVCJ,388,0.992268,18.0,24.541237,38.0,250,0.007732,`ftol` termination condition is satisfied.,`xtol` termination condition is satisfied.,Both `ftol` and `xtol` termination conditions ...


## 7) Spread-relative diagnostics (test and train)

Using option-level quotes, we compute per-snapshot:
- fraction of quotes priced within **half-spread** and within the **full spread**
- mean \(|error|/spread\)
- sMAPE and RMSE normalized by mean market premium

We plot these over time and summarize them with bootstrap CIs.


In [9]:
spread_metric_timeseries = analysis.spread_metric_timeseries


def spread_metric_summary_table(opt_metrics_df, currency, split="test", n_boot=3000):
    return analysis.spread_metric_summary_table(opt_metrics_df, currency, split=split, n_boot=n_boot, rng=RNG)


## 8) Error breakdowns by moneyness and maturity (test set)

We report **MAE** broken down by:
- absolute log-moneyness \(|\log(K/F)|\) buckets
- maturity buckets (based on time-to-maturity in years)

Bucket metrics are computed **within each snapshot**, then averaged across snapshots (equal-weighted over time).


In [10]:
MONEY_BINS = analysis.MONEY_BINS
MONEY_LABELS = analysis.MONEY_LABELS
T_BINS = analysis.T_BINS
T_LABELS = analysis.T_LABELS
bucket_mae_by_snapshot = analysis.bucket_mae_by_snapshot
bucket_all = state.bucket_all

display(bucket_all.head(6))


,snapshot_ts,bucket,mae,model,split,bucket_type,currency
0,2025-12-30 17:31:15+00:00,|log(K/F)|≤0.05,0.003093,Black,test,moneyness,BTC
1,2025-12-30 17:31:15+00:00,0.05–0.15,0.003357,Black,test,moneyness,BTC
2,2025-12-30 17:31:15+00:00,0.15–0.30,0.004060,Black,test,moneyness,BTC
3,2025-12-30 17:31:15+00:00,>0.30,0.004229,Black,test,moneyness,BTC
4,2025-12-30 21:17:51+00:00,|log(K/F)|≤0.05,0.003925,Black,test,moneyness,BTC
5,2025-12-30 21:17:51+00:00,0.05–0.15,0.003334,Black,test,moneyness,BTC


In [11]:
def bucket_summary_table(bucket_df, currency, bucket_type, n_boot=2000):
    return analysis.bucket_summary_table(bucket_df, currency, bucket_type, n_boot=n_boot, rng=RNG)


plot_bucket_bars = analysis.plot_bucket_bars


## 9) Parameter stability and bound-pressure diagnostics

We provide:
- time-series plots for key parameters (Heston and SVCJ),
- distribution boxplots,
- “near-bound” rates using the calibration bounds (in natural parameter space),
- and the Heston/SVCJ **Feller ratio** \(\sigma_v^2/(2\kappa\theta)\) as a constraint-pressure proxy.


In [12]:
RHO_LB = analysis.RHO_LB
RHO_UB = analysis.RHO_UB
BOUNDS = analysis.BOUNDS
EPS = analysis.EPS
add_feller_ratio = analysis.add_feller_ratio
near_bound_rates = analysis.near_bound_rates
results_ok = add_feller_ratio(results_ok)


In [13]:
plot_param_timeseries = analysis.plot_param_timeseries
plot_param_boxplots = analysis.plot_param_boxplots


## 10) A single “report runner” per currency

To keep the notebook readable, we wrap the repeated steps into one function that:
- prints key counts,
- displays error time series + boxplots (train & test),
- outputs error summary tables (train & test),
- outputs spread-relative diagnostics (train & test),
- outputs bucket plots (test),
- outputs convergence diagnostics (already global),
- outputs parameter stability and near-bound tables.


In [14]:
def run_currency_report(currency, n_boot=3000):
    return analysis.run_currency_report(state, currency, n_boot=n_boot)


RUN_REPORTS = True
N_BOOT = 3000

if RUN_REPORTS:
    run_currency_report("BTC", n_boot=N_BOOT)
    run_currency_report("ETH", n_boot=N_BOOT)
else:
    print("RUN_REPORTS=False. Set RUN_REPORTS=True to generate the full BTC/ETH report outputs.")


REPORT — BTC
Snapshots: 388
Success rates:


,success_rate
model,
Black,1.000000
Heston,1.000000
SVCJ,0.984536


Summary table — BTC / train


,currency,split,metric,item,n,mean,ci_95,median,q25,q75,std,min,max,win_rate
0,BTC,train,MAE,Black,388,0.003585,"[0.00350767, 0.00367332]",0.003526,0.003016,0.004043,0.000830,0.001887,0.011707,NaN
1,BTC,train,MAE,GAIN: Black→Heston (%),388,0.584574,"[0.568736, 0.600585]",0.569619,0.472271,0.683543,0.158433,0.088807,0.893843,1.000000
2,BTC,train,MAE,GAIN: Black→Heston (abs),388,0.002153,"[0.00205929, 0.00224494]",0.001893,0.001525,0.002786,0.000931,0.000244,0.008569,1.000000
3,BTC,train,MAE,GAIN: Heston→SVCJ (%),382,0.311310,"[0.296763, 0.325077]",0.313654,0.235180,0.398070,0.145142,-0.395210,0.713238,0.963918
4,BTC,train,MAE,GAIN: Heston→SVCJ (abs),382,0.000459,"[0.000430977, 0.000486388]",0.000462,0.000259,0.000598,0.000276,-0.000422,0.001399,0.963918
5,BTC,train,MAE,Heston,388,0.001432,"[0.0013781, 0.00148207]",0.001458,0.001096,0.001716,0.000526,0.000457,0.003515,NaN
6,BTC,train,MAE,SVCJ,382,0.000963,"[0.000924797, 0.00100359]",0.000923,0.000694,0.001189,0.000391,0.000243,0.003151,NaN
7,BTC,train,RMSE,Black,388,0.005268,"[0.00515292, 0.00539136]",0.005155,0.004413,0.005968,0.001191,0.002757,0.016335,NaN
8,BTC,train,RMSE,GAIN: Black→Heston (%),388,0.495770,"[0.474226, 0.517092]",0.458135,0.344106,0.603261,0.215568,-0.009419,0.907387,0.997423
9,BTC,train,RMSE,GAIN: Black→Heston (abs),388,0.002698,"[0.00254052, 0.00285091]",0.002186,0.001556,0.003394,0.001589,-0.000037,0.010782,0.997423


Summary table — BTC / test


...

,currency,bucket_type,bucket,model,n,mean,ci_95,median,q25,q75
2,ETH,maturity,1m–3m,Black,388,0.003561,"[0.00345498, 0.00367157]",0.003379,0.002852,0.004031
6,ETH,maturity,1m–3m,Heston,388,0.002157,"[0.00204124, 0.00228102]",0.002010,0.001311,0.002807
10,ETH,maturity,1m–3m,SVCJ,385,0.001647,"[0.00154937, 0.00175481]",0.001401,0.000819,0.002215
1,ETH,maturity,1w–1m,Black,388,0.002823,"[0.00272675, 0.00293017]",0.002701,0.002123,0.003501
5,ETH,maturity,1w–1m,Heston,388,0.001516,"[0.00143332, 0.00160818]",0.001391,0.000870,0.001972
9,ETH,maturity,1w–1m,SVCJ,385,0.001164,"[0.00109164, 0.0012418]",0.000929,0.000630,0.001535
3,ETH,maturity,>3m,Black,388,0.006405,"[0.0060715, 0.00680377]",0.005516,0.004286,0.007499
7,ETH,maturity,>3m,Heston,388,0.003041,"[0.00289537, 0.00319969]",0.002940,0.001952,0.003914
11,ETH,maturity,>3m,SVCJ,385,0.002257,"[0.00212172, 0.00239844]",0.002086,0.001223,0.002961
0,ETH,maturity,≤1w,Black,385,0.002362,"[0.00223361, 0.0025036]",0.002056,0.001414,0.002923


Parameter stability — Heston


,model,param,lb,ub,near_lb_rate,near_ub_rate,min,q25,median,q75,max
0,Heston,kappa,0.000100,50.000000,0.000000,0.154639,2.327755,12.419123,19.877498,34.264432,50.000000
1,Heston,rho,-0.999909,0.999909,0.000000,0.000000,-0.551716,-0.253818,-0.216242,-0.177514,-0.077753
2,Heston,sigma_v,0.000100,10.000000,0.000000,0.000000,1.725326,3.519097,4.445530,5.768856,7.757770
3,Heston,theta,0.000001,5.000000,0.000000,0.000000,0.408677,0.465652,0.500984,0.535161,0.638419
4,Heston,v0,0.000001,5.000000,0.007732,0.000000,0.057155,0.288184,0.481619,0.598945,2.334085


Parameter stability — SVCJ


,model,param,lb,ub,near_lb_rate,near_ub_rate,min,q25,median,q75,max
0,SVCJ,ell_v,0.000001,10.000000,0.441558,0.070130,0.005621,0.160694,0.228676,1.091447,10.000000
1,SVCJ,ell_y,-5.000000,5.000000,0.000000,0.000000,-0.537190,-0.226784,-0.151798,-0.088366,0.401741
2,SVCJ,kappa,0.000100,50.000000,0.000000,0.148052,1.480626,6.289775,13.340471,32.660093,50.000000
3,SVCJ,lam,0.000001,10.000000,0.005195,0.020779,0.157044,1.046303,1.747463,3.023569,9.996884
4,SVCJ,rho,-0.999909,0.999909,0.111688,0.000000,-0.999610,-0.078741,0.134715,0.281619,0.783435
5,SVCJ,rho_j,-0.999909,0.999909,0.537662,0.000000,-0.999908,-0.998997,-0.998934,-0.020001,0.905024
6,SVCJ,sigma_v,0.000100,10.000000,0.096104,0.000000,0.000942,1.473949,2.181769,3.428237,6.236144
7,SVCJ,sigma_y,0.000100,5.000000,0.893506,0.000000,0.000109,0.000277,0.000277,0.001006,0.272497
8,SVCJ,theta,0.000001,5.000000,0.109091,0.000000,0.005924,0.208124,0.267696,0.315256,0.422822
9,SVCJ,v0,0.000001,5.000000,0.010390,0.000000,0.044627,0.239100,0.377613,0.478189,2.193851


## 11) Export thesis artifacts (tables + figures)

This cell saves:
- tables into `./tables/`
- figures into `./figs/` as HTML (always) and PNG (if Kaleido is available)

Set `EXPORT = True` to activate.


In [15]:
EXPORT = False

OUT_TABLES = Path("tables")
OUT_FIGS = Path("figs")


def _safe_write_image(fig, path_png):
    try:
        fig.write_image(path_png, scale=2)
        return True
    except Exception as exc:
        print(f"[warn] Could not write PNG (needs kaleido): {path_png}\n  {exc}")
        return False


if EXPORT:
    OUT_TABLES.mkdir(parents=True, exist_ok=True)
    OUT_FIGS.mkdir(parents=True, exist_ok=True)

    conv = convergence_table(results_long)
    conv.to_csv(OUT_TABLES / "convergence_table.csv", index=False)

    for currency in results_long["currency"].unique():
        for split in ["train", "test"]:
            tbl = error_summary_table(results_long, currency, split=split)
            tbl.to_csv(OUT_TABLES / f"{currency.lower()}_{split}_error_summary.csv", index=False)

            tbl_sp = spread_metric_summary_table(opt_metrics, currency, split=split)
            tbl_sp.to_csv(OUT_TABLES / f"{currency.lower()}_{split}_spread_summary.csv", index=False)

            fig_ts = plot_error_timeseries(results_long, currency, split=split)
            fig_ts.write_html(OUT_FIGS / f"{currency.lower()}_{split}_errors_timeseries.html")
            _safe_write_image(fig_ts, OUT_FIGS / f"{currency.lower()}_{split}_errors_timeseries.png")

            fig_box = plot_error_boxplots(results_long, currency, split=split)
            fig_box.write_html(OUT_FIGS / f"{currency.lower()}_{split}_errors_boxplots.html")
            _safe_write_image(fig_box, OUT_FIGS / f"{currency.lower()}_{split}_errors_boxplots.png")

            fig_sp = spread_metric_timeseries(opt_metrics, currency, split=split)
            fig_sp.write_html(OUT_FIGS / f"{currency.lower()}_{split}_spread_timeseries.html")
            _safe_write_image(fig_sp, OUT_FIGS / f"{currency.lower()}_{split}_spread_timeseries.png")

        b1 = bucket_summary_table(bucket_all, currency, "moneyness")
        b2 = bucket_summary_table(bucket_all, currency, "maturity")
        b1.to_csv(OUT_TABLES / f"{currency.lower()}_test_bucket_moneyness.csv", index=False)
        b2.to_csv(OUT_TABLES / f"{currency.lower()}_test_bucket_maturity.csv", index=False)

        fig_bm = plot_bucket_bars(bucket_all, currency, "moneyness")
        fig_bt = plot_bucket_bars(bucket_all, currency, "maturity")
        fig_bm.write_html(OUT_FIGS / f"{currency.lower()}_test_bucket_moneyness.html")
        fig_bt.write_html(OUT_FIGS / f"{currency.lower()}_test_bucket_maturity.html")
        _safe_write_image(fig_bm, OUT_FIGS / f"{currency.lower()}_test_bucket_moneyness.png")
        _safe_write_image(fig_bt, OUT_FIGS / f"{currency.lower()}_test_bucket_maturity.png")

        nb_hes = near_bound_rates(results_long[results_long["currency"] == currency], "Heston")
        nb_svcj = near_bound_rates(results_long[results_long["currency"] == currency], "SVCJ")
        nb_hes.to_csv(OUT_TABLES / f"{currency.lower()}_heston_near_bound_rates.csv", index=False)
        nb_svcj.to_csv(OUT_TABLES / f"{currency.lower()}_svcj_near_bound_rates.csv", index=False)

    print("Export complete.")
else:
    print("EXPORT=False (no files written). Set EXPORT=True to save tables/figures.")


EXPORT=False (no files written). Set EXPORT=True to save tables/figures.
